<a href="https://colab.research.google.com/github/nghff/vlm-pca-exercise/blob/main/Raphi_Task.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install --upgrade vllm

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.modules import padding
from torch.utils.data._utils.collate import default_collate

from transformers import AutoTokenizer, AutoProcessor, AutoModelForImageTextToText
from PIL import Image, UnidentifiedImageError

import io, os, base64, requests
import regex as re
from typing import List
import numpy as np
import pandas as pd
from hashlib import sha256
import sys

import time
from tqdm import tqdm as tqdm

import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/'
SAVE_DIR = os.path.join(DRIVE_DIR, 'MyDrive/Raphi_Task_Data')
if not os.path.exists(SAVE_DIR):
  os.makedirs(SAVE_DIR)

DATA_PORTION = 1

model_id = "Qwen/Qwen3-VL-2B-Instruct"
#model_id = "llava-hf/llava-1.5-7b-hf"
#model_id = "allenai/Molmo2-8B"

RESULTS_SAVE_DIR = os.path.join(SAVE_DIR, model_id)
if not os.path.exists(RESULTS_SAVE_DIR):
  os.makedirs(RESULTS_SAVE_DIR)

In [ ]:
def save_df(name, df, show_df=False):
    df_path = os.path.join(RESULTS_SAVE_DIR, f'{name}.pkl')
    df.to_pickle(df_path)

    if show_df:
        display(df)

def load_df(name):
    df_path = os.path.join(RESULTS_SAVE_DIR, f'{name}.pkl')
    return pd.read_pickle(df_path)

### Dataset implementation for pixmo dataset

In [ ]:
def pixmo_collate_fn(processor, timeout=10, max_length: int = 4096):
    def collate_fn(batch):
        messages_list = [b["messages"] for b in batch]
        targets = torch.tensor([b["target_count"] for b in batch])
        bad_messages_idx = []

        for i in range(len(messages_list)):
            for message in messages_list[i]:
                if message["role"] != "user":
                    continue
                for content in message["content"]:
                    if content["type"] != "image":
                        continue
                    img_url = content["image"]

                    if img_url.startswith("http://") or img_url.startswith("https://"):
                        try:
                            r = requests.get(img_url, timeout=timeout, allow_redirects=True)
                            r.raise_for_status()
                        except:
                            print(f"request failed: {img_url}")
                            bad_messages_idx.append(i)
                            break
                        ct = (r.headers.get("content-type") or "").lower()

                        if "image" not in ct:
                            print(f"filtered: {img_url}")
                            bad_messages_idx.append(i)
                            break
                    else:
                        print(f"filtered: {img_url}")
                        bad_messages_idx.append(i)
                        break
                break

        # remove messages with failed images
        for idx in sorted(bad_messages_idx, reverse=True):
            targets = torch.cat([targets[:idx], targets[idx+1:]])
            del messages_list[idx]

        print(f'num messages: {len(messages_list)}')

        inputs = processor.apply_chat_template(
            messages_list,
            tokenize=True,
            continue_final_message=True,
            return_dict=True,
            return_tensors="pt",
            padding='longest'
        )

        inputs["target_count"] = targets
        return inputs
    return collate_fn

class pixmoDataset(Dataset):
  splits = {
    'val': 'validation-00000-of-00001.parquet',
    'train': 'train-00000-of-00001.parquet',
    'test': 'test-00000-of-00001.parquet'
  }

  def __init__(self,
               split: str,
               processor: AutoProcessor,
               portion=1.0, max_length=4096,
               **kwargs):
    if split == "validation":
      split = "val"
    if split not in pixmoDataset.splits:
      raise ValueError("'split' not recognized in dataset constructor")

    split_file = pixmoDataset.splits[split]
    self.data = pd.read_parquet("hf://datasets/allenai/pixmo-count/data/" + split_file)
    self.data = self.data.filter(items=['image_url', 'label', 'count'])
    self.processor = processor
    self.portion = portion
    self.max_length = max_length
    if 'filter_fn' in kwargs:
      self.data = kwargs['filter_fn'](self.data)

  def __len__(self):
    return int(len(self.data) * self.portion)

  def __getitem__(self, idx):
        row = self.data.iloc[idx]
        label = row["label"]
        image = row["image_url"]
        target = int(row["count"])

        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image},
                    {"type": "text", "text": f"How many {label} are in this photo?"},
                ],
            },
            {
                "role": "assistant",
                "content": [{"type": "text", "text": f"The number of {label} is: "}],
            },
        ]

        return {"messages": messages, "target_count": target}

### Evaluator

In [ ]:
# define metrics
def sanitize_pairs(pred: List[str], target: List[str]):
  p = []
  t = []
  misfits = []
  for i in range(len(pred)):
    if pred[i] != '':
      p.append(int(re.sub(r'[^\d]+', '', pred[i])))
      t.append(target[i])
    else:
      misfits.append((pred[i], target[i]))
  return p, t, misfits

def accuracy(pred: List[int], target: List[int]):
  return np.mean(pred == target)

def mean_error(pred: List[int], target: List[int]):
  return np.mean(np.abs(pred - target))

def mean_squared_error(pred: List[int], target: List[int]):
  return np.mean((pred - target)**2)

In [ ]:
class Evaluator:
  def __init__(self,
               metric_fns={},
               batch_size=4,
               shuffle=False,
               collate_fn=None,
               token_allowance=2,
               callbacks=[]):
    self.metric_fns = metric_fns
    self.batch_size = batch_size
    self.shuffle = shuffle
    self.collate_fn = collate_fn
    self.max_new_tokens = token_allowance
    self.callbacks = callbacks

  def __call__(self, model, processor, dataset, **kwargs):
    return self.evaluate(model, processor, dataset, **kwargs)

  def evaluate(self, model, processor, dataset, **kwargs):
    dataloader = DataLoader(dataset, batch_size=self.batch_size, shuffle=self.shuffle, collate_fn=self.collate_fn)
    all_preds = []
    all_targets = []
    all_inputs = []
    all_misfits = []
    dl_progress = tqdm(dataloader)

    model.eval()
    with torch.inference_mode():
        for batch in dl_progress:
            target_counts = batch.pop('target_count')
            target_counts = [count.item() for count in target_counts]

            inputs = batch
            inputs = {k: v.to(model.device, non_blocking=True) for k, v in inputs.items()}

            output = model.generate(
                **inputs,
                max_new_tokens=self.max_new_tokens,
                num_beams=1,
                do_sample=False,
                return_dict_in_generate=True,
                output_scores=False,
                output_hidden_states=kwargs.get('needs_hidden_states', False),
                output_attentions=kwargs.get('needs_attentions', False)
            )

            generated_ids_trimmed = [
                out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs['input_ids'], output['sequences'])
            ]
            generated_strs = processor.batch_decode(
                generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
            )

            generated_preds, target_counts, misfits = sanitize_pairs(generated_strs, target_counts)

            all_preds.extend(generated_preds)
            all_targets.extend(target_counts)
            all_inputs.extend([input_ids for input_ids in inputs['input_ids']])
            all_misfits.extend(misfits)

            print(f'preds: {generated_preds}')
            print(f'targets: {target_counts}')
            for callback in self.callbacks:
                callback(
                    inputs,
                    target_counts,
                    generated_preds,
                    output['hidden_states'] if 'hidden_states' in output else None,
                    output['attentions'] if 'attentions' in output else None
                )

    metrics = {}
    for name, fn in metric_fns.items():
        metrics[name] = fn(np.array(all_preds), np.array(all_targets))

    return {
        'metrics': metrics,
        'all_inputs': all_inputs,
        'all_preds': all_preds,
        'all_targets': all_targets,
        'all_misfits': all_misfits
    }

## Evaluation

### Run evaluation

In [ ]:
load_params = {
    'Qwen/Qwen3-VL-2B-Instruct': {
        'dtype': torch.bfloat16,
        'device_map': "auto"
    },

    'llava-hf/llava-1.5-7b-hf': {
        'torch_dtype': torch.float16,
        'low_cpu_mem_usage': True
    },

    'allenai/Molmo2-8B': {
        'dtype': "auto",
        'device_map': "auto",
        'trust_remote_code': True
    }
}

model = AutoModelForImageTextToText.from_pretrained(
     model_id, **load_params[model_id]
)

processor = AutoProcessor.from_pretrained(model_id, trust_remote_code = True)
processor.tokenizer.padding_side = "left"

In [ ]:
test_dataset = pixmoDataset(split='test', processor=processor, portion=DATA_PORTION)
print(len(test_dataset))

metric_fns = {
    "accuracy": accuracy,
    "mean_error": mean_error,
    "mean_squared_error": mean_squared_error
}

In [ ]:
evaluator = Evaluator(
    metric_fns=metric_fns,
    batch_size=4,
    shuffle=False,
    collate_fn=pixmo_collate_fn(processor)
)
results = evaluator(model, processor, test_dataset)

# show misfits
misfits_df = pd.DataFrame(results['all_misfits'], columns=['pred', 'target'])
save_df('all_misfits', misfits_df, show_df=True)

# display results
results_df = pd.DataFrame(results['metrics'], index=[0])
save_df('results', results_df, show_df=True)
results_df

# save all input-pred-target pairs
all_df = pd.DataFrame(zip(results['all_inputs'], results['all_preds'], results['all_targets']), columns=['input', 'pred', 'target'])
save_df('all_input_pred_target_pairs', all_df, show_df=False)

## PCA Analysis

In [ ]:
load_params = {
    'Qwen/Qwen3-VL-2B-Instruct': {
        'dtype': torch.bfloat16,
        'device_map': "auto",
        'attn_implementation': 'eager'
    },

    'llava-hf/llava-1.5-7b-hf': {
        'torch_dtype': torch.float16,
        'low_cpu_mem_usage': True,
        'attn_implementation': 'eager'
    },

    'allenai/Molmo2-8B': {
        'dtype': "auto",
        'device_map': "auto",
        'trust_remote_code': True
    }
}

model = AutoModelForImageTextToText.from_pretrained(
     model_id, **load_params[model_id]
).to('cuda:0')
processor = AutoProcessor.from_pretrained(model_id, trust_remote_code = True)
processor.tokenizer.padding_side = "left"

### Logging

In [ ]:
layers_to_hook = {
    'Qwen/Qwen3-VL-2B-Instruct': [
        'model.visual.blocks.1',                # early visual features extracted by visual encoder
        'model.visual.blocks.11',                # mid visual features extracted by visual encoder
        'model.visual.blocks.23',               # final visual information extracted from image by visual encoder

        'model.visual.merger',                  # visual features 'mapped' to global language space

        'model.visual.deepstack_merger_list.0', # visual features added to corresponding activations during early language reasoning (in the 'language' decoder)
        'model.visual.deepstack_merger_list.1', # visual features added to corresponding activations during early language reasoning (in the 'language' decoder)
        'model.visual.deepstack_merger_list.2', # visual features added to corresponding activations during early language reasoning (in the 'language' decoder)

        'model.language_model.layers.0',
        'model.language_model.layers.1',        # early language features extracted by decoder
        'model.language_model.layers.13',       # mid language features extracted by decoder
        'model.language_model.layers.20',
        'model.language_model.layers.27'       # final language features extracted by decoder
    ],
    'llava-hf/llava-1.5-7b-hf': [
        'model.vision_tower.vision_model.encoder.layers.1',
        'model.vision_tower.vision_model.encoder.layers.11',
        'model.vision_tower.vision_model.encoder.layers.23',

        'model.multi_modal_projector',

        'model.language_model.layers.0',
        'model.language_model.layers.1',
        'model.language_model.layers.15',
        'model.language_model.layers.23',
        'model.language_model.layers.31'
    ],
    'allenai/Molmo2-8B':[
        'model.vision_backbone.image_vit.transformer.resblocks.1',
        'model.vision_backbone.image_vit.transformer.resblocks.12',
        'model.vision_backbone.image_vit.transformer.resblocks.24',

        'model.vision_backbone.image_projector',

        'model.transformer.blocks.0',
        'model.transformer.blocks.1',
        'model.transformer.blocks.17',
        'model.transformer.blocks.26',
        'model.transformer.blocks.35'
    ]
}

attentions_to_grab = {
    'Qwen/Qwen3-VL-2B-Instruct': [
        0, 1, 13, 27
    ],
    'llava-hf/llava-1.5-7b-hf': [
        0, 1, 15, 30, 31
    ],
    'allenai/Molmo2-8B':[
        # eager attention not supported
    ]
}

In [ ]:
# activation cache
activation_cache = {}

# activation data
activation_data = []

# hook for activation logging
def log_activations(module_name):
    def hook(module, inp, out):
        print(f'caught {module_name}')
        x = out[0] if isinstance(out, (tuple, list)) else out
        if torch.is_tensor(x):
            activation_cache[module_name] = (
                x.detach()
                .to(dtype=torch.float32)
                .cpu()
            )
    return hook

# attach hooks
def attach_hooks(model, model_id):
    to_hook = layers_to_hook[model_id]
    for module_name, module in model.named_modules():
        if module_name in to_hook:
            print(f'hooked: {module_name}')
            module.register_forward_hook(log_activations(module_name))

# callback for storing activation data
def store_activations(inputs, targets, preds, hidden_states, attentions):
    print('detaching activations')
    attentions = attentions[0] # assumes single token generated

    input_ids_cpu = inputs["input_ids"].detach().cpu().tolist()
    tokens_batch = [processor.tokenizer.convert_ids_to_tokens(ids) for ids in input_ids_cpu]

    act_cpu = {layer: activation_cache[layer].detach().cpu().numpy() for layer in layers_to_hook[model_id]}

    att_cpu = {lid: attentions[lid].detach().cpu().to(torch.float16).numpy() for lid in attentions_to_grab[model_id]}

    print('detached data, storing')
    for i in range(len(targets)):
        print(f'storing idx {i} in batch')
        activation_data.append({
            'input_tokens': tokens_batch[i],
            'target': targets[i],
            'pred': preds[i]
        })
        for layer in layers_to_hook[model_id]:
            print(f'storing {layer}')
            activation_data[-1][layer] = act_cpu[layer][i]

        if attentions_to_grab[model_id] != []:
            for layer_id in attentions_to_grab[model_id]:
                print(f'storing attention_{layer_id}')
                activation_data[-1][f'attention_{layer_id}'] = att_cpu[layer_id][i].mean(axis=-3)

# clear hooks
def clear_hooks():
    for module_name, module in model.named_modules():
        if module_name in layers_to_hook[model_id]:
            module._forward_hooks.clear()

# reset logs
def reset_logs():
    activation_cache.clear()
    activation_data.clear()
    clear_hooks()

# pca plotting
def layer_pca_plots():
  pass

### Run feedforward

In [ ]:
# function for filtering out rows in pixmo with count > 5

def counts_filter(data: pd.DataFrame):
    return data.iloc[(data['count'].to_numpy() <= 5).tolist()].reset_index(drop=True)

In [ ]:
test_dataset = pixmoDataset(split='test', processor=processor, portion=DATA_PORTION, filter_fn=counts_filter)

print(f'test dataset size: {len(test_dataset)}')
print(test_dataset.data.head(10))

metric_fns = {}

In [ ]:
reset_logs()

evaluator = Evaluator(
    metric_fns=metric_fns,
    batch_size=4,
    shuffle=False,
    collate_fn=pixmo_collate_fn(processor),
    token_allowance=1, # 1 thru 9 represented by uno token,
    callbacks=[store_activations]
)

attach_hooks(model, model_id)

results = evaluator(model, processor, test_dataset, needs_attentions=True)

save_df('activation_data', pd.DataFrame(activation_data), show_df=True)

clear_hooks()

In [ ]:
pd.DataFrame(activation_data).head()

In [ ]:

print('a')
load_df('activation_data')['attention_1'][0]

In [ ]:
activation_data[0]['attention_1'].shape

In [ ]:
#show
plt.imshow(activation_data[0]['attention_1'], vmin=0, vmax=0.01)

In [ ]:
for name, module in model.model.transformer.blocks[0].self_attn.k_norm.named_modules():
    print(name)

In [ ]:
print(len([name for name, module in model.named_modules()]))

In [ ]:
for name, module in model.named_modules():
  print(name)

In [ ]:
model